# DiffusionGemma 26B-A4B-it on Kaggle dual T4

Official weights: `google/diffusiongemma` (Transformers, ~52 GB BF16).
Two Tesla T4s are **16 GB each**. This notebook shards FP16 weights with `device_map="auto"` and a CPU offload budget (the path that actually generated on Kaggle). NF4/bitsandbytes can load but hits a RoPE shape error on this architecture, so it is not the default.

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-U", "transformers", "accelerate", "bitsandbytes", "kagglehub",
])
print("pip ok")

In [ ]:
import json, os, time, traceback
from pathlib import Path

import torch

print("torch", torch.__version__, "cuda", torch.version.cuda)
print("cuda_available", torch.cuda.is_available())
print("gpu_count", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"gpu{i}", p.name, f"{p.total_memory/1024**3:.2f} GiB")

assert torch.cuda.is_available() and torch.cuda.device_count() >= 1, "need a GPU"
if torch.cuda.device_count() < 2:
    print("WARNING: expected 2x T4; continuing with", torch.cuda.device_count())

In [ ]:
import transformers
from transformers import AutoProcessor, BitsAndBytesConfig, DiffusionGemmaForBlockDiffusion

print("transformers", transformers.__version__)
print("class", DiffusionGemmaForBlockDiffusion)

CANDIDATES = [
    "/kaggle/input/models/google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
    "/kaggle/input/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
    "/kaggle/input/diffusiongemma/transformers/diffusiongemma-26b-a4b-it",
    "/kaggle/input/google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it/1",
]
MODEL_ID = next((p for p in CANDIDATES if os.path.exists(p)), None)
if MODEL_ID is None:
    import kagglehub
    MODEL_ID = kagglehub.model_download("google/diffusiongemma/transformers/diffusiongemma-26b-a4b-it")
print("MODEL_ID", MODEL_ID)
print("contents", sorted(os.listdir(MODEL_ID))[:30])

In [ ]:
def max_mem():
    n = torch.cuda.device_count()
    mem = {i: "15GiB" for i in range(n)}
    mem["cpu"] = "24GiB"
    return mem

def try_load(model_id):
    processor = AutoProcessor.from_pretrained(model_id)
    strategies = [
        ("fp16-offload",
         dict(
             torch_dtype=torch.float16,
             device_map="auto",
             max_memory=max_mem(),
             low_cpu_mem_usage=True,
         )),
    ]
    errors = []
    for name, kwargs in strategies:
        print("load strategy", name, flush=True)
        t0 = time.perf_counter()
        try:
            model = DiffusionGemmaForBlockDiffusion.from_pretrained(model_id, **kwargs)
            print("loaded", name, "in", round(time.perf_counter() - t0, 1), "s", flush=True)
            return processor, model, name, errors
        except Exception as e:
            msg = f"{name}: {type(e).__name__}: {e}"
            errors.append(msg)
            print(msg, flush=True)
            traceback.print_exc()
            torch.cuda.empty_cache()
    raise RuntimeError("all load strategies failed:\n" + "\n".join(errors))

processor, model, load_name, load_errors = try_load(MODEL_ID)
print("using", load_name)
print("device_map sample", list(getattr(model, "hf_device_map", {}).items())[:12])

In [ ]:
prompt = [{"role": "user", "content": "Explain in 4 short sentences what discrete diffusion language models do."}]
inputs = processor.apply_chat_template(
    prompt,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
)
device = next(model.parameters()).device
inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
print("input_ids", tuple(inputs["input_ids"].shape), "device", device, flush=True)

t0 = time.perf_counter()
with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=256)
elapsed = time.perf_counter() - t0
raw = processor.decode(output[0], skip_special_tokens=False)
if isinstance(raw, list):
    raw = raw[0]
text = str(raw).replace("<pad>", "").strip()
print("generate_s", round(elapsed, 2))
print(text)

In [ ]:
mem = {}
for i in range(torch.cuda.device_count()):
    mem[f"gpu{i}_alloc_gib"] = round(torch.cuda.memory_allocated(i) / 1024**3, 3)
    mem[f"gpu{i}_reserved_gib"] = round(torch.cuda.memory_reserved(i) / 1024**3, 3)

results = {
    "model_id": MODEL_ID,
    "load_strategy": load_name,
    "load_errors": load_errors,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "gpu_count": torch.cuda.device_count(),
    "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
    "generate_s": round(elapsed, 3),
    "prompt": prompt[0]["content"],
    "text": text,
    "memory": mem,
}
out = Path("/kaggle/working/results.json")
out.write_text(json.dumps(results, indent=2, ensure_ascii=False))
Path("/kaggle/working/generation.txt").write_text(str(text))
print("wrote", out)
print(json.dumps({k: results[k] for k in ("load_strategy", "gpu_count", "gpus", "generate_s", "memory")}, indent=2))